# 07 - Community sentiment and topic analysis

**Research sub-question.** Once notebook 04 has assigned authors to reply-network communities, and notebook 05 has assigned sentiment and topics to comments, what does each community actually sound like?

This notebook is downstream-only. It does **not** recompute networks, sentiment models, or topic models. It joins persisted outputs and turns them into a defensible community atlas: first validating coverage, then profiling interpretable communities, then producing report-ready findings and caveats.

## 1 - Setup

Outputs are written under `data/processed/07_community_sentiment_topics/` and `plots/`. This keeps the notebook separate from the upstream network, sentiment, topic, diffusion, and platform-comparison stages.

In [ ]:
from __future__ import annotations

import json
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = PROJECT_ROOT / "data"
PROCESSED = DATA / "processed"
PREPROCESSED = PROCESSED / "01_preprocessed"
SENTIMENT_TOPICS = PROCESSED / "03_sentiment_topics"
NETWORKS = PROCESSED / "04_networks"
COMMUNITY_OUT = PROCESSED / "07_community_sentiment_topics"
PLOTS = PROJECT_ROOT / "plots"

COMMUNITY_OUT.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)

MIN_COMMENTS = 30
FIGURE_MIN_COMMENTS = 100
STRICT_MIN_COMMENTS = 250
REPORT_GRADE_MIN_COMMENTS = FIGURE_MIN_COMMENTS
REPORT_GRADE_MAX_BINOMIAL_HALF_WIDTH_PCT = 10.0
TOP_COMMUNITIES_PER_PLATFORM = 12

REDDIT_COLOR = "#3b6fa1"
YOUTUBE_COLOR = "#c45a3d"
NEUTRAL_COLOR = "#777777"
NEGATIVE_COLOR = "#b5524a"
POSITIVE_COLOR = "#4f8f63"

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {PROJECT_ROOT}")
print(f"Outputs    : {COMMUNITY_OUT}")


## 2 - Inputs and join contract

Required inputs:

| Stage | File | Join keys | Purpose |
|---|---|---|---|
| Preprocessing | `01_preprocessed/{reddit,youtube}_comments.csv` | base table | comment metadata, text, author hash |
| Networks | `04_networks/{reddit,youtube}_community_assignments.csv` | `platform, author_hash` | Louvain community, role, PageRank, community frame |
| Sentiment | `03_sentiment_topics/sentiment_comment_level.csv` | `platform, comment_id` | transformer headline sentiment plus VADER baseline |
| Topics | `03_sentiment_topics/topics_lda_platform_dominant_per_comment_labelled.csv` | `platform, comment_id` | dominant platform-specific LDA topic |
| Optional topics | `03_sentiment_topics/topics_lda_platform_doc_topic_long.csv` | `platform, comment_id` | full topic weights for sensitivity checks |

The network community assignment is author-level. Comments by authors who are not reply-graph nodes will not receive a community label. That is expected and is reported as coverage rather than treated as an error.

In [ ]:
def read_required(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")
    return pd.read_csv(path, low_memory=False)


def require_columns(df: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def first_existing(df: pd.DataFrame, candidates: list[str], default=pd.NA) -> pd.Series:
    for col in candidates:
        if col in df.columns:
            return df[col]
    return pd.Series(default, index=df.index)


def pct(part: float, whole: float) -> float:
    return float(part / whole * 100) if whole else float("nan")


def entropy_from_counts(counts: pd.Series) -> float:
    values = counts.astype(float).to_numpy()
    total = values.sum()
    if total <= 0:
        return float("nan")
    probs = values / total
    return float(-(probs * np.log2(probs)).sum())


def compact_text(value: object, width: int = 220) -> str:
    text = str(value or "").replace("\n", " ").strip()
    text = " ".join(text.split())
    return textwrap.shorten(text, width=width, placeholder="...") if text else ""


def safe_mode(series: pd.Series):
    values = series.dropna()
    return values.mode().iloc[0] if not values.empty else pd.NA


def representative_comments(g: pd.DataFrame, n: int = 3) -> str:
    cols = ["comment_id", "topic_label", "sentiment_label", "sentiment_score", "topic_probability"]
    available = [c for c in cols if c in g.columns]
    sample = (
        g.dropna(subset=["clean_text"])
        .assign(_len=lambda d: d["clean_text"].astype(str).str.len())
        .query("_len >= 40")
        .sort_values(["topic_probability", "_len"], ascending=[False, False])
        .head(n)
    )
    records = []
    for _, row in sample.iterrows():
        item = {c: row[c] for c in available}
        item["text"] = compact_text(row.get("clean_text"), width=220)
        records.append(item)
    return json.dumps(records, ensure_ascii=False)


def label_for_plot(row: pd.Series) -> str:
    scope = row.get("dominant_scope_by_comments")
    if pd.isna(scope):
        scope = row.get("community_dominant_scope")
    scope = str(scope) if not pd.isna(scope) else "unknown"
    return f"{row['platform']} c{int(row['community'])} | {scope} | n={int(row['n_comments'])}"


def atlas_label(row: pd.Series) -> str:
    scope = row.get("dominant_scope_by_comments")
    scope = str(scope) if not pd.isna(scope) else "unknown"
    return f"{row['platform']}_c{int(row['community'])} | {scope} | n={int(row['n_comments'])}"


def wrap_label(value: object, width: int = 42) -> str:
    return "\n".join(textwrap.wrap(str(value), width=width, break_long_words=False))

## 3 - Load base tables

YouTube comments do not always carry human-readable channel titles, so this cell attaches `channel_title` from `youtube_videos.csv` when available. The unified `scope` column means subreddit for Reddit and channel title/channel id for YouTube.

In [ ]:
reddit_comments = read_required(PREPROCESSED / "reddit_comments.csv")
youtube_comments = read_required(PREPROCESSED / "youtube_comments.csv")

videos_path = PREPROCESSED / "youtube_videos.csv"
if videos_path.exists():
    videos = pd.read_csv(videos_path)
    if {"video_id", "channel_title"}.issubset(videos.columns):
        youtube_comments = youtube_comments.merge(
            videos[["video_id", "channel_title"]].drop_duplicates(),
            on="video_id",
            how="left",
        )
else:
    videos = pd.DataFrame()

reddit_comments["scope"] = reddit_comments.get("subreddit")
youtube_comments["scope"] = first_existing(youtube_comments, ["channel_title", "channel_id"])

base_cols = [
    "platform", "comment_id", "author_hash", "date", "clean_text", "vader_text",
    "token_count", "char_count", "scope", "subreddit", "submission_id", "video_id",
    "channel_id", "channel_title", "is_reply", "depth", "score", "like_count",
]
for df in (reddit_comments, youtube_comments):
    for col in base_cols:
        if col not in df.columns:
            df[col] = pd.NA

comments = pd.concat(
    [reddit_comments[base_cols], youtube_comments[base_cols]],
    ignore_index=True,
)
require_columns(comments, ["platform", "comment_id", "author_hash", "scope"], "comments")

print(f"Reddit comments : {len(reddit_comments):,}")
print(f"YouTube comments: {len(youtube_comments):,}")
print(f"Combined        : {len(comments):,}")
comments.head(3)

## 4 - Load communities, sentiment, and topics

In [ ]:
assignments = pd.concat(
    [
        read_required(NETWORKS / "reddit_community_assignments.csv"),
        read_required(NETWORKS / "youtube_community_assignments.csv"),
    ],
    ignore_index=True,
)
require_columns(
    assignments,
    ["platform", "author_hash", "community", "role", "pagerank", "community_dominant_scope", "community_top_frame"],
    "community assignments",
)

sentiment = read_required(SENTIMENT_TOPICS / "sentiment_comment_level.csv")
require_columns(sentiment, ["platform", "comment_id"], "sentiment")
sent_cols = [
    "platform", "comment_id", "tx_label", "tx_compound", "headline_sentiment_label",
    "headline_sentiment_score", "vader_label", "vader_compound", "lex_label", "lex_score",
]
sent_cols = [c for c in sent_cols if c in sentiment.columns]
sentiment = sentiment[sent_cols].drop_duplicates(["platform", "comment_id"])

topics = read_required(SENTIMENT_TOPICS / "topics_lda_platform_dominant_per_comment_labelled.csv")
require_columns(topics, ["platform", "comment_id", "topic_label", "topic_probability"], "dominant topics")
topic_cols = [
    "platform", "comment_id", "platform_topic_id", "platform_topic", "topic_label",
    "topic_title", "platform_topic_label", "topic_probability", "topic_weight",
]
topic_cols = [c for c in topic_cols if c in topics.columns]
topics = topics[topic_cols].drop_duplicates(["platform", "comment_id"])

full_topic_weights_path = SENTIMENT_TOPICS / "topics_lda_platform_doc_topic_long.csv"
full_topic_weights_available = full_topic_weights_path.exists()

print(f"Community assignments: {len(assignments):,} authors")
print(f"Sentiment rows       : {len(sentiment):,}")
print(f"Dominant-topic rows  : {len(topics):,}")
print(f"Full topic weights   : {full_topic_weights_available}")

## 5 - Build the comment-level analytic table

This is the safest hand-off artifact for any downstream notebook. It keeps one row per comment and carries both author-level community information and comment-level sentiment/topic labels.

In [ ]:
joined = comments.merge(
    assignments,
    on=["platform", "author_hash"],
    how="left",
    validate="many_to_one",
)
joined = joined.merge(sentiment, on=["platform", "comment_id"], how="left", validate="one_to_one")
joined = joined.merge(topics, on=["platform", "comment_id"], how="left", validate="one_to_one")

joined["has_community"] = joined["community"].notna()
joined["sentiment_label"] = first_existing(joined, ["headline_sentiment_label", "tx_label", "vader_label"])
joined["sentiment_score"] = first_existing(joined, ["headline_sentiment_score", "tx_compound", "vader_compound"])
joined["sentiment_method"] = np.select(
    [
        joined.get("headline_sentiment_label", pd.Series(pd.NA, index=joined.index)).notna(),
        joined.get("tx_label", pd.Series(pd.NA, index=joined.index)).notna(),
        joined.get("vader_label", pd.Series(pd.NA, index=joined.index)).notna(),
    ],
    ["Transformer headline", "Transformer", "VADER"],
    default="missing",
)
joined["community"] = joined["community"].astype("Int64")
joined["community_key"] = np.where(
    joined["has_community"],
    joined["platform"].astype(str) + "_c" + joined["community"].astype(str),
    pd.NA,
)
joined["analytic_ready"] = joined["has_community"] & joined["sentiment_label"].notna() & joined["topic_label"].notna()

out_path = COMMUNITY_OUT / "comment_community_sentiment_topics.csv"
joined.to_csv(out_path, index=False)
print(f"Wrote {out_path.relative_to(PROJECT_ROOT)} ({len(joined):,} rows)")
joined.head(3)

# Quality gate

Before interpreting communities, check whether the join is strong enough. The key distinction is:

- **Coverage:** how much of each platform can be assigned to a reply-network community.
- **Interpretability:** how many assigned communities have enough comments to support meaningful summaries.

## 6 - Coverage audit

This is the first table to quote before any community-level result. It makes the YouTube limitation explicit: many YouTube commenters are top-level-only users and therefore not nodes in the reply graph.

In [ ]:
coverage_rows = []
for platform, g in joined.groupby("platform"):
    total_comments = len(g)
    total_authors = g["author_hash"].nunique()
    comments_with_community = int(g["has_community"].sum())
    authors_with_community = g.loc[g["has_community"], "author_hash"].nunique()
    comments_with_sentiment = int(g["sentiment_label"].notna().sum())
    comments_with_topic = int(g["topic_label"].notna().sum())
    analytic_comments = int(g["analytic_ready"].sum())
    coverage_rows.append({
        "platform": platform,
        "total_comments": total_comments,
        "comments_with_community": comments_with_community,
        "pct_comments_with_community": pct(comments_with_community, total_comments),
        "total_authors": total_authors,
        "authors_with_community": authors_with_community,
        "pct_authors_with_community": pct(authors_with_community, total_authors),
        "comments_with_sentiment": comments_with_sentiment,
        "comments_with_topic": comments_with_topic,
        "analytic_comments": analytic_comments,
        "pct_analytic_comments": pct(analytic_comments, total_comments),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(COMMUNITY_OUT / "community_coverage_summary.csv", index=False)
coverage

In [ ]:
funnel_rows = []
for _, row in coverage.iterrows():
    platform = row["platform"]
    stages = [
        ("All comments", row["total_comments"]),
        ("With sentiment", row["comments_with_sentiment"]),
        ("With topic", row["comments_with_topic"]),
        ("With community", row["comments_with_community"]),
        ("Analytic-ready", row["analytic_comments"]),
    ]
    for order, (stage, count) in enumerate(stages):
        funnel_rows.append({
            "platform": platform,
            "stage": stage,
            "stage_order": order,
            "comments": int(count),
            "pct_total": pct(count, row["total_comments"]),
        })

coverage_funnel = pd.DataFrame(funnel_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, platform, color in zip(axes, ["reddit", "youtube"], [REDDIT_COLOR, YOUTUBE_COLOR]):
    data = coverage_funnel[coverage_funnel["platform"] == platform].sort_values("stage_order")
    ax.plot(data["stage_order"], data["pct_total"], marker="o", linewidth=2.2, color=color)
    ax.fill_between(data["stage_order"], data["pct_total"], color=color, alpha=0.12)
    ax.set_xticks(data["stage_order"])
    ax.set_xticklabels(data["stage"], rotation=25, ha="right")
    ax.set_ylim(0, 105)
    ax.set_title(f"{platform.title()} analytic coverage funnel")
    ax.set_ylabel("% of all comments")
    for _, point in data.iterrows():
        ax.text(point["stage_order"], point["pct_total"] + 2, f"{point['pct_total']:.1f}%", ha="center", fontsize=8)
plt.tight_layout()
out = PLOTS / "community_coverage_funnel.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print(f"Wrote {out.relative_to(PROJECT_ROOT)}")

fig, ax = plt.subplots(figsize=(7, 4))
coverage_plot = coverage.melt(
    id_vars="platform",
    value_vars=["pct_comments_with_community", "pct_authors_with_community", "pct_analytic_comments"],
    var_name="coverage_type",
    value_name="pct",
)
coverage_plot["coverage_type"] = coverage_plot["coverage_type"].map({
    "pct_comments_with_community": "Comments with community",
    "pct_authors_with_community": "Authors with community",
    "pct_analytic_comments": "Analytic-ready comments",
})
sns.barplot(
    data=coverage_plot,
    x="platform",
    y="pct",
    hue="coverage_type",
    ax=ax,
    palette=[REDDIT_COLOR, NEUTRAL_COLOR, YOUTUBE_COLOR],
)
ax.set_ylabel("%")
ax.set_xlabel("")
ax.set_ylim(0, 100)
ax.set_title("Community join coverage by platform")
ax.legend(title="", loc="upper right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=2, fontsize=8)
plt.tight_layout()
out = PLOTS / "community_join_coverage.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print(f"Wrote {out.relative_to(PROJECT_ROOT)}")

# Community atlas

The atlas starts from a one-row-per-community summary, then adds topic mix, author-normalized sentiment, and report candidates. Use these tables before making any substantive claims about “what a community thinks.”

## 7 - Community-level sentiment and topic summary

This table aggregates only comments that have community, sentiment, and dominant-topic labels. It is the base table for the atlas.

In [ ]:
analytic = joined[joined["analytic_ready"]].copy()

summary_rows = []
for (platform, community), g in analytic.groupby(["platform", "community"], dropna=True):
    topic_counts = g["topic_label"].value_counts()
    sentiment_counts = g["sentiment_label"].value_counts(normalize=True)
    scope_counts = g["scope"].value_counts(dropna=True)
    role_counts = g.drop_duplicates("author_hash")["role"].value_counts(dropna=True)
    author_frame = g.drop_duplicates("author_hash")
    n_comments = len(g)
    n_authors = g["author_hash"].nunique()
    dominant_topic = topic_counts.index[0] if len(topic_counts) else pd.NA
    dominant_scope = scope_counts.index[0] if len(scope_counts) else pd.NA
    dominant_role = role_counts.index[0] if len(role_counts) else pd.NA
    summary_rows.append({
        "platform": platform,
        "community": int(community),
        "community_key": f"{platform}_c{int(community)}",
        "n_comments": n_comments,
        "n_authors": n_authors,
        "dominant_scope_by_comments": dominant_scope,
        "dominant_scope_comment_pct": pct(scope_counts.iloc[0], n_comments) if len(scope_counts) else float("nan"),
        "community_dominant_scope": safe_mode(g["community_dominant_scope"]),
        "community_top_frame": safe_mode(g["community_top_frame"]),
        "dominant_role": dominant_role,
        "dominant_topic": dominant_topic,
        "dominant_topic_pct": pct(topic_counts.iloc[0], n_comments) if len(topic_counts) else float("nan"),
        "topic_entropy": entropy_from_counts(topic_counts),
        "mean_topic_probability": float(g["topic_probability"].mean()),
        "mean_sentiment": float(g["sentiment_score"].mean()),
        "median_sentiment": float(g["sentiment_score"].median()),
        "pct_negative": float(sentiment_counts.get("negative", 0.0) * 100),
        "pct_neutral": float(sentiment_counts.get("neutral", 0.0) * 100),
        "pct_positive": float(sentiment_counts.get("positive", 0.0) * 100),
        "mean_vader_compound": float(g["vader_compound"].mean()) if "vader_compound" in g else float("nan"),
        "mean_author_pagerank": float(author_frame["pagerank"].mean()),
        "max_author_pagerank": float(author_frame["pagerank"].max()),
        "representative_comments": representative_comments(g, n=3),
    })

community_summary = pd.DataFrame(summary_rows)
platform_totals = analytic.groupby("platform").size().rename("platform_analytic_comments")
community_summary = community_summary.merge(platform_totals, on="platform", how="left")
community_summary["pct_platform_analytic_comments"] = (
    community_summary["n_comments"] / community_summary["platform_analytic_comments"] * 100
)
community_summary["reportable_min_comments"] = community_summary["n_comments"] >= MIN_COMMENTS
community_summary["figure_min_comments"] = community_summary["n_comments"] >= FIGURE_MIN_COMMENTS
community_summary["strict_min_comments"] = community_summary["n_comments"] >= STRICT_MIN_COMMENTS
community_summary["plot_label"] = community_summary.apply(label_for_plot, axis=1)
community_summary["atlas_label"] = community_summary.apply(atlas_label, axis=1)
community_summary = community_summary.sort_values(["platform", "n_comments"], ascending=[True, False])

community_summary.to_csv(COMMUNITY_OUT / "community_sentiment_topic_summary.csv", index=False)
community_summary[community_summary["reportable_min_comments"]].head(12)

## 8 - Community size threshold audit

Raw Louvain community counts are not inherently interpretable. This cell shows how many communities survive practical size thresholds and how much assigned discussion they cover.

In [ ]:
threshold_rows = []
thresholds = [1, MIN_COMMENTS, FIGURE_MIN_COMMENTS, STRICT_MIN_COMMENTS]
for platform, g in community_summary.groupby("platform"):
    total_communities = len(g)
    total_comments = g["n_comments"].sum()
    for threshold in thresholds:
        subset = g[g["n_comments"] >= threshold]
        threshold_rows.append({
            "platform": platform,
            "min_comments": threshold,
            "communities": len(subset),
            "pct_communities": pct(len(subset), total_communities),
            "comments": int(subset["n_comments"].sum()),
            "pct_assigned_comments": pct(subset["n_comments"].sum(), total_comments),
            "max_95pct_binomial_half_width_pct": float(1.96 * np.sqrt(0.25 / threshold) * 100),
            "chosen_for_report": threshold == REPORT_GRADE_MIN_COMMENTS,
            "threshold_role": (
                "all_assigned" if threshold == 1 else
                "exploratory" if threshold == MIN_COMMENTS else
                "report_grade" if threshold == FIGURE_MIN_COMMENTS else
                "strict_examples"
            ),
        })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.to_csv(COMMUNITY_OUT / "community_threshold_summary.csv", index=False)

threshold_justification = threshold_summary.copy()
threshold_justification["statistical_rationale"] = np.where(
    threshold_justification["chosen_for_report"],
    "Chosen report-grade cutoff: at least 100 comments gives a worst-case 95% binomial half-width below 10 percentage points while retaining useful platform coverage.",
    "Sensitivity threshold retained for coverage/interpretability comparison.",
)
threshold_justification.to_csv(COMMUNITY_OUT / "community_threshold_justification.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharex=True)
for ax, metric, title in zip(
    axes,
    ["communities", "pct_assigned_comments"],
    ["Communities retained", "Assigned comments retained"],
):
    sns.lineplot(
        data=threshold_summary,
        x="min_comments",
        y=metric,
        hue="platform",
        marker="o",
        palette={"reddit": REDDIT_COLOR, "youtube": YOUTUBE_COLOR},
        ax=ax,
    )
    ax.axvline(REPORT_GRADE_MIN_COMMENTS, color="black", linestyle="--", linewidth=0.8, alpha=0.65)
    ax.set_xscale("log")
    ax.set_xlabel("Minimum comments per community")
    ax.set_title(title)
    if metric == "pct_assigned_comments":
        ax.set_ylabel("% of community-assigned comments")
        ax.set_ylim(0, 105)
    else:
        ax.set_ylabel("Communities")
    ax.legend(title="")
plt.tight_layout()
out = PLOTS / "community_threshold_audit.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print(f"Wrote {out.relative_to(PROJECT_ROOT)}")
threshold_summary

## 9 - Topic distribution within communities

This long table is the safest source for heatmaps and statements like “community X is 47% war-risk topic and 38% market-reaction topic.” Prefer it over dominant topic alone.

In [ ]:
topic_dist = (
    analytic.groupby(["platform", "community", "community_key", "topic_label"], dropna=False)
    .agg(
        n_comments=("comment_id", "count"),
        n_authors=("author_hash", "nunique"),
        mean_sentiment=("sentiment_score", "mean"),
        pct_negative=("sentiment_label", lambda s: float((s == "negative").mean() * 100)),
        mean_topic_probability=("topic_probability", "mean"),
    )
    .reset_index()
)
community_sizes = community_summary[
    ["platform", "community", "n_comments", "plot_label", "atlas_label", "reportable_min_comments", "figure_min_comments"]
].rename(columns={"n_comments": "community_comments"})
topic_dist = topic_dist.merge(community_sizes, on=["platform", "community"], how="left")
topic_dist["pct_within_community"] = topic_dist["n_comments"] / topic_dist["community_comments"] * 100
topic_dist = topic_dist.sort_values(
    ["platform", "community_comments", "community", "n_comments"],
    ascending=[True, False, True, False],
)
topic_dist.to_csv(COMMUNITY_OUT / "community_topic_distribution.csv", index=False)
topic_dist.head(12)

## 9b - Expected topic shares from full topic weights

Dominant-topic counts are easy to read, but they discard uncertainty. When the full LDA document-topic table is available, this section aggregates expected topic weight by community. Use this table for nuanced interpretation and keep dominant-topic labels for compact summaries.

In [ ]:
topic_label_lookup = (
    topics[["platform", "platform_topic", "topic_label"]]
    .drop_duplicates(["platform", "platform_topic"])
)

if full_topic_weights_available:
    full_topic_weights = read_required(full_topic_weights_path)
    require_columns(full_topic_weights, ["platform", "comment_id", "platform_topic", "topic_weight"], "full topic weights")
    expected_topic_base = (
        analytic[["platform", "comment_id", "community", "community_key"]]
        .merge(full_topic_weights, on=["platform", "comment_id"], how="inner")
        .merge(topic_label_lookup, on=["platform", "platform_topic"], how="left")
    )
    expected_topic_dist = (
        expected_topic_base.groupby(["platform", "community", "community_key", "topic_label"], dropna=False)
        .agg(
            expected_topic_weight=("topic_weight", "sum"),
            mean_topic_weight=("topic_weight", "mean"),
            n_comment_topic_rows=("comment_id", "count"),
        )
        .reset_index()
    )
    community_n = analytic.groupby(["platform", "community"]).size().rename("community_comments").reset_index()
    expected_topic_dist = expected_topic_dist.merge(community_n, on=["platform", "community"], how="left")
    expected_topic_dist["expected_pct_within_community"] = expected_topic_dist["expected_topic_weight"] / expected_topic_dist["community_comments"] * 100
    expected_topic_dist = expected_topic_dist.sort_values(
        ["platform", "community_comments", "community", "expected_pct_within_community"],
        ascending=[True, False, True, False],
    )
else:
    expected_topic_dist = pd.DataFrame(columns=[
        "platform", "community", "community_key", "topic_label", "expected_topic_weight",
        "mean_topic_weight", "n_comment_topic_rows", "community_comments",
        "expected_pct_within_community",
    ])

expected_topic_dist.to_csv(COMMUNITY_OUT / "community_expected_topic_distribution.csv", index=False)

def expected_topic_mix_for(platform: str, community: int, n: int = 3) -> str:
    data = expected_topic_dist[
        (expected_topic_dist["platform"] == platform)
        & (expected_topic_dist["community"] == community)
    ]
    if data.empty:
        return topic_mix_for(platform, community, n=n) if "topic_mix_for" in globals() else ""
    data = data.sort_values("expected_pct_within_community", ascending=False).head(n)
    return "; ".join(
        f"{row['topic_label']} ({row['expected_pct_within_community']:.1f}%)"
        for _, row in data.iterrows()
    )

expected_topic_dist.head(12)

## 10 - Author-normalized sentiment sensitivity

Comment-level means can be dominated by prolific authors. This table averages sentiment within each author-community pair first, then averages authors within each community. Use this to check whether a community-level sentiment result survives author normalization.

In [ ]:
author_level = (
    analytic.groupby(["platform", "community", "community_key", "author_hash"], dropna=True)
    .agg(
        author_comments=("comment_id", "count"),
        author_mean_sentiment=("sentiment_score", "mean"),
        author_pct_negative=("sentiment_label", lambda s: float((s == "negative").mean() * 100)),
        role=("role", safe_mode),
        pagerank=("pagerank", "max"),
    )
    .reset_index()
)
author_norm = (
    author_level.groupby(["platform", "community", "community_key"], dropna=True)
    .agg(
        n_authors=("author_hash", "nunique"),
        author_normalized_mean_sentiment=("author_mean_sentiment", "mean"),
        author_normalized_pct_negative=("author_pct_negative", "mean"),
        median_author_comments=("author_comments", "median"),
        max_author_comments=("author_comments", "max"),
    )
    .reset_index()
)
author_norm = author_norm.merge(
    community_summary[[
        "platform", "community", "n_comments", "mean_sentiment", "pct_negative",
        "plot_label", "atlas_label", "reportable_min_comments", "figure_min_comments",
    ]],
    on=["platform", "community"],
    how="left",
)
author_norm["mean_sentiment_delta_author_minus_comment"] = (
    author_norm["author_normalized_mean_sentiment"] - author_norm["mean_sentiment"]
)
author_norm.to_csv(COMMUNITY_OUT / "community_author_normalized_sentiment.csv", index=False)
author_norm[author_norm["reportable_min_comments"]].sort_values(
    ["platform", "n_comments"], ascending=[True, False]
).head(12)

## 10b - Author-level bootstrap uncertainty

Community sentiment is estimated from model-labelled comments. This bootstrap resamples authors within each report-grade community, so intervals are less dominated by prolific commenters than a comment-level bootstrap.

In [ ]:
def bootstrap_author_metrics(g: pd.DataFrame, dominant_topic: str, n_boot: int = 400, seed: int = 0) -> dict:
    author_metrics = (
        g.groupby("author_hash")
        .agg(
            mean_sentiment=("sentiment_score", "mean"),
            pct_negative=("sentiment_label", lambda s: float((s == "negative").mean() * 100)),
            topic_share=("topic_label", lambda s: float((s == dominant_topic).mean() * 100)),
        )
        .dropna()
    )
    if author_metrics.empty:
        return {
            "mean_sentiment_lo": np.nan, "mean_sentiment_hi": np.nan,
            "pct_negative_lo": np.nan, "pct_negative_hi": np.nan,
            "dominant_topic_share_lo": np.nan, "dominant_topic_share_hi": np.nan,
            "author_mean_sentiment_center": np.nan,
            "author_pct_negative_center": np.nan,
            "author_dominant_topic_share_center": np.nan,
            "bootstrap_authors": 0,
        }
    centers = author_metrics.mean(numeric_only=True)
    rng = np.random.default_rng(seed)
    values = author_metrics.to_numpy()
    draws = rng.integers(0, len(values), size=(n_boot, len(values)))
    samples = values[draws].mean(axis=1)
    return {
        "mean_sentiment_lo": float(np.percentile(samples[:, 0], 2.5)),
        "mean_sentiment_hi": float(np.percentile(samples[:, 0], 97.5)),
        "pct_negative_lo": float(np.percentile(samples[:, 1], 2.5)),
        "pct_negative_hi": float(np.percentile(samples[:, 1], 97.5)),
        "dominant_topic_share_lo": float(np.percentile(samples[:, 2], 2.5)),
        "dominant_topic_share_hi": float(np.percentile(samples[:, 2], 97.5)),
        "author_mean_sentiment_center": float(centers["mean_sentiment"]),
        "author_pct_negative_center": float(centers["pct_negative"]),
        "author_dominant_topic_share_center": float(centers["topic_share"]),
        "bootstrap_authors": int(len(values)),
    }


uncertainty_rows = []
figure_communities = community_summary[community_summary["figure_min_comments"]].copy()
for _, row in figure_communities.iterrows():
    g = analytic[
        (analytic["platform"] == row["platform"])
        & (analytic["community"] == row["community"])
    ]
    metrics = bootstrap_author_metrics(
        g,
        dominant_topic=row["dominant_topic"],
        n_boot=400,
        seed=int(row["community"]) + (0 if row["platform"] == "reddit" else 10_000),
    )
    uncertainty_rows.append({
        "platform": row["platform"],
        "community": int(row["community"]),
        "community_key": row["community_key"],
        "plot_label": row["plot_label"],
        "n_comments": int(row["n_comments"]),
        "n_authors": int(row["n_authors"]),
        "mean_sentiment": float(row["mean_sentiment"]),
        "pct_negative": float(row["pct_negative"]),
        "dominant_topic": row["dominant_topic"],
        "dominant_topic_pct": float(row["dominant_topic_pct"]),
        **metrics,
    })

community_uncertainty = pd.DataFrame(uncertainty_rows)
community_uncertainty.to_csv(COMMUNITY_OUT / "community_bootstrap_uncertainty.csv", index=False)

plot_uncertainty = (
    community_uncertainty.sort_values(["platform", "n_comments"], ascending=[True, False])
    .groupby("platform", group_keys=False)
    .head(10)
    .sort_values(["platform", "pct_negative"])
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharex=True)
for ax, platform, color in zip(axes, ["reddit", "youtube"], [REDDIT_COLOR, YOUTUBE_COLOR]):
    data = plot_uncertainty[plot_uncertainty["platform"] == platform].copy()
    if data.empty:
        ax.set_title(f"{platform.title()} - no report-grade communities")
        continue
    labels = [wrap_label(label, 36) for label in data["plot_label"]]
    xerr = np.vstack([
        data["author_pct_negative_center"] - data["pct_negative_lo"],
        data["pct_negative_hi"] - data["author_pct_negative_center"],
    ])
    xerr = np.clip(xerr, 0, None)
    ax.errorbar(data["author_pct_negative_center"], labels, xerr=xerr, fmt="o", color=color, ecolor="grey", capsize=3)
    ax.set_title(f"{platform.title()} negative share with author-bootstrap 95% CI")
    ax.set_xlabel("% negative comments")
    ax.set_ylabel("")
plt.tight_layout()
out = PLOTS / "community_negative_share_ci.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()

community_uncertainty.head(12)

## 11 - Report candidate communities and atlas table

This table is the interpretability layer. It narrows the full community table to larger communities and adds readable labels, top-topic mixtures, sentiment composition, and author-normalized sensitivity. Use this before opening the full joined comment table.

In [ ]:
def topic_mix_for(platform: str, community: int, n: int = 3, *, use_expected: bool = True) -> str:
    if use_expected and "expected_topic_dist" in globals() and not expected_topic_dist.empty:
        data = expected_topic_dist[
            (expected_topic_dist["platform"] == platform)
            & (expected_topic_dist["community"] == community)
        ].sort_values("expected_pct_within_community", ascending=False).head(n)
        if not data.empty:
            return "; ".join(
                f"{row['topic_label']} ({row['expected_pct_within_community']:.1f}%)"
                for _, row in data.iterrows()
            )
    data = topic_dist[(topic_dist["platform"] == platform) & (topic_dist["community"] == community)]
    data = data.sort_values("pct_within_community", ascending=False).head(n)
    return "; ".join(
        f"{row['topic_label']} ({row['pct_within_community']:.1f}%)"
        for _, row in data.iterrows()
    )


def enrich_community_table(base: pd.DataFrame) -> pd.DataFrame:
    enriched = base.merge(
        author_norm[[
            "platform", "community", "author_normalized_mean_sentiment",
            "author_normalized_pct_negative", "mean_sentiment_delta_author_minus_comment",
            "median_author_comments", "max_author_comments",
        ]],
        on=["platform", "community"],
        how="left",
    )
    enriched = enriched.merge(
        community_uncertainty[[
            "platform", "community", "mean_sentiment_lo", "mean_sentiment_hi",
            "pct_negative_lo", "pct_negative_hi", "dominant_topic_share_lo",
            "dominant_topic_share_hi", "author_mean_sentiment_center",
            "author_pct_negative_center", "author_dominant_topic_share_center",
            "bootstrap_authors",
        ]],
        on=["platform", "community"],
        how="left",
    )
    enriched["topic_mix_top3"] = enriched.apply(
        lambda row: topic_mix_for(row["platform"], int(row["community"]), n=3),
        axis=1,
    )
    enriched["sentiment_mix"] = enriched.apply(
        lambda row: f"negative {row['pct_negative']:.1f}%, neutral {row['pct_neutral']:.1f}%, positive {row['pct_positive']:.1f}%",
        axis=1,
    )
    enriched["interpretation_label"] = enriched["atlas_label"]
    enriched["author_concentration_pct"] = enriched["max_author_comments"] / enriched["n_comments"] * 100
    enriched["topic_mix_basis"] = np.where(
        "expected_topic_dist" in globals() and not expected_topic_dist.empty,
        "expected_topic_weights",
        "dominant_topic_counts",
    )
    return enriched


def readiness(row: pd.Series) -> tuple[str, str]:
    if row["n_comments"] < MIN_COMMENTS:
        return "below_threshold", "Too few comments for interpretation."
    if row["n_comments"] < REPORT_GRADE_MIN_COMMENTS:
        return "exploratory", "Enough for inspection, not for headline figures."
    warnings = []
    if row["platform"] == "youtube":
        warnings.append("YouTube community coverage is partial.")
    if row["author_concentration_pct"] > 15:
        warnings.append("One author contributes more than 15% of comments.")
    if abs(row["mean_sentiment_delta_author_minus_comment"]) > 0.05:
        warnings.append("Author-normalized sentiment shifts by more than 0.05.")
    if row["dominant_topic_pct"] < 40:
        warnings.append("Dominant topic is weak; topic mix is broad.")
    if warnings:
        return "conditional", " ".join(warnings)
    return "strong", "Large, diverse-enough community with stable author-normalized sentiment."


report_grade_base = community_summary[community_summary["n_comments"] >= REPORT_GRADE_MIN_COMMENTS].copy()
community_report_grade_subset = enrich_community_table(report_grade_base)
readiness_values = community_report_grade_subset.apply(readiness, axis=1)
community_report_grade_subset["interpretation_readiness"] = [value[0] for value in readiness_values]
community_report_grade_subset["readiness_reason"] = [value[1] for value in readiness_values]
community_report_grade_subset.to_csv(COMMUNITY_OUT / "community_report_grade_subset.csv", index=False)

report_grade_keys = community_report_grade_subset[["platform", "community"]].drop_duplicates()
comment_community_report_grade_subset = analytic.merge(
    report_grade_keys.assign(report_grade_community=True),
    on=["platform", "community"],
    how="inner",
)
comment_community_report_grade_subset.to_csv(COMMUNITY_OUT / "comment_community_report_grade_subset.csv", index=False)

report_candidates = (
    community_report_grade_subset
    .sort_values(["platform", "n_comments"], ascending=[True, False])
    .groupby("platform", group_keys=False)
    .head(20)
)
report_candidates.to_csv(COMMUNITY_OUT / "community_report_candidates.csv", index=False)

community_atlas = report_candidates.copy()
community_atlas = community_atlas[[
    "platform", "community", "community_key", "interpretation_label", "interpretation_readiness",
    "readiness_reason", "n_comments", "n_authors", "dominant_scope_by_comments",
    "dominant_scope_comment_pct", "community_top_frame", "dominant_role",
    "dominant_topic", "dominant_topic_pct", "topic_mix_top3", "topic_mix_basis",
    "topic_entropy", "mean_topic_probability", "sentiment_mix", "mean_sentiment",
    "mean_sentiment_lo", "mean_sentiment_hi", "pct_negative",
    "author_pct_negative_center", "pct_negative_lo", "pct_negative_hi",
    "author_normalized_mean_sentiment",
    "author_normalized_pct_negative", "mean_sentiment_delta_author_minus_comment",
    "author_concentration_pct", "median_author_comments", "max_author_comments",
    "bootstrap_authors", "representative_comments",
]]
community_atlas.to_csv(COMMUNITY_OUT / "community_atlas.csv", index=False)

print(
    "Report-grade subset:",
    community_report_grade_subset.groupby("platform").agg(
        communities=("community", "count"),
        comments=("n_comments", "sum"),
        min_comments=("n_comments", "min"),
        min_authors=("n_authors", "min"),
    ).to_string()
)

display_cols = [
    "interpretation_label", "interpretation_readiness", "readiness_reason",
    "topic_mix_top3", "sentiment_mix", "pct_negative_lo", "pct_negative_hi",
]
community_atlas[display_cols].head(12)

## 11b - Community value beyond subreddit/channel scope

Many network communities are dominated by a subreddit or YouTube channel. This diagnostic compares each community to its dominant scope baseline. Large deltas indicate that the reply-network community adds information beyond simply grouping by subreddit/channel.

In [ ]:
scope_summary = (
    analytic.groupby(["platform", "scope"], dropna=False)
    .agg(
        scope_comments=("comment_id", "count"),
        scope_authors=("author_hash", "nunique"),
        scope_mean_sentiment=("sentiment_score", "mean"),
        scope_pct_negative=("sentiment_label", pct_negative),
        scope_dominant_topic=("topic_label", safe_mode),
    )
    .reset_index()
    .rename(columns={"scope": "dominant_scope_by_comments"})
)

community_scope_baseline = community_summary.merge(
    scope_summary,
    on=["platform", "dominant_scope_by_comments"],
    how="left",
)
community_scope_baseline["mean_sentiment_delta_vs_scope"] = (
    community_scope_baseline["mean_sentiment"] - community_scope_baseline["scope_mean_sentiment"]
)
community_scope_baseline["pct_negative_delta_vs_scope"] = (
    community_scope_baseline["pct_negative"] - community_scope_baseline["scope_pct_negative"]
)
community_scope_baseline["community_topic_differs_from_scope"] = (
    community_scope_baseline["dominant_topic"] != community_scope_baseline["scope_dominant_topic"]
)
community_scope_baseline.to_csv(COMMUNITY_OUT / "community_scope_baseline_comparison.csv", index=False)

plot_scope_delta = (
    community_scope_baseline[community_scope_baseline["figure_min_comments"]]
    .assign(abs_delta=lambda d: d["mean_sentiment_delta_vs_scope"].abs())
    .sort_values(["platform", "abs_delta"], ascending=[True, False])
    .groupby("platform", group_keys=False)
    .head(10)
    .sort_values(["platform", "mean_sentiment_delta_vs_scope"])
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)
for ax, platform, color in zip(axes, ["reddit", "youtube"], [REDDIT_COLOR, YOUTUBE_COLOR]):
    data = plot_scope_delta[plot_scope_delta["platform"] == platform]
    if data.empty:
        ax.set_title(f"{platform.title()} - no report-grade communities")
        continue
    labels = [wrap_label(label, 35) for label in data["plot_label"]]
    ax.barh(labels, data["mean_sentiment_delta_vs_scope"], color=color, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.6)
    ax.set_title(f"{platform.title()} community sentiment delta vs dominant scope")
    ax.set_xlabel("Community mean sentiment - scope mean sentiment")
    ax.set_ylabel("")
plt.tight_layout()
out = PLOTS / "community_vs_scope_sentiment_delta.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()

community_scope_baseline[
    community_scope_baseline["figure_min_comments"]
][[
    "platform", "community", "n_comments", "dominant_scope_by_comments",
    "mean_sentiment", "scope_mean_sentiment", "mean_sentiment_delta_vs_scope",
    "dominant_topic", "scope_dominant_topic", "community_topic_differs_from_scope",
]].head(12)

## 12 - Report figure: community atlas

The report uses the bubble atlas, not the auxiliary topic heatmaps or sentiment-profile bars. The kept figure shows only report-grade communities after the 100-comment threshold.


In [ ]:
atlas_plot = community_summary[community_summary["figure_min_comments"]].copy()
topic_order = atlas_plot["dominant_topic"].value_counts().index.tolist()
topic_palette = dict(zip(topic_order, sns.color_palette("tab10", n_colors=max(3, len(topic_order)))))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), sharey=True)
for ax, platform in zip(axes, ["reddit", "youtube"]):
    data = atlas_plot[atlas_plot["platform"] == platform].copy()
    if data.empty:
        ax.set_title(f"{platform.title()} - no communities above figure threshold")
        continue
    sns.scatterplot(
        data=data,
        x="mean_sentiment",
        y="topic_entropy",
        size="n_comments",
        hue="dominant_topic",
        sizes=(70, 720),
        alpha=0.78,
        palette=topic_palette,
        ax=ax,
        legend=False,
    )
    label_data = data.sort_values("n_comments", ascending=False).head(5)
    for _, row in label_data.iterrows():
        ax.text(
            row["mean_sentiment"] + 0.01,
            row["topic_entropy"],
            f"c{int(row['community'])}",
            fontsize=8,
            va="center",
        )
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.45)
    ax.set_title(f"{platform.title()} community atlas")
    ax.set_xlabel("Mean transformer sentiment")
    ax.set_ylabel("Topic entropy (higher = more mixed)")
plt.tight_layout()
out = PLOTS / "community_bubble_atlas.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print(f"Wrote {out.relative_to(PROJECT_ROOT)}")

# Finding-first close

The final cells turn the atlas into explicit claims. This avoids over-reading small communities, under-covered YouTube comments, or dominant-topic shortcuts.

## 13 - Report-ready findings and claim safety

Use **report-grade communities with >=100 comments** for headline figures. This threshold is chosen because it keeps the worst-case 95 % binomial half-width at **<=9.8 percentage points**, while still retaining enough YouTube reply-network communities to interpret. Use `MIN_COMMENTS` communities only for exploratory appendices or qualitative inspection, and use the stricter >=250 subset only for selective large-community examples.


In [ ]:
def top_row(platform: str, sort_col: str, ascending: bool = False, min_comments: int = FIGURE_MIN_COMMENTS) -> pd.Series:
    data = community_summary[
        (community_summary["platform"] == platform)
        & (community_summary["n_comments"] >= min_comments)
    ]
    if data.empty:
        return pd.Series(dtype=object)
    return data.sort_values(sort_col, ascending=ascending).iloc[0]


for platform in ["reddit", "youtube"]:
    cov = coverage.set_index("platform").loc[platform]
    threshold = threshold_summary[
        (threshold_summary["platform"] == platform)
        & (threshold_summary["min_comments"] == FIGURE_MIN_COMMENTS)
    ].iloc[0]
    largest = top_row(platform, "n_comments", min_comments=FIGURE_MIN_COMMENTS)
    most_negative = top_row(platform, "pct_negative", min_comments=FIGURE_MIN_COMMENTS)
    most_mixed = top_row(platform, "topic_entropy", min_comments=FIGURE_MIN_COMMENTS)
    print(f"\n{platform.upper()}")
    print(f"- Join coverage: {int(cov['comments_with_community']):,}/{int(cov['total_comments']):,} comments ({cov['pct_comments_with_community']:.1f}%) and {cov['pct_authors_with_community']:.1f}% of authors.")
    print(f"- Report-grade threshold: {int(threshold['communities'])} communities with >= {FIGURE_MIN_COMMENTS} comments, covering {threshold['pct_assigned_comments']:.1f}% of community-assigned comments.")
    if not largest.empty:
        print(f"- Largest report-grade community: c{int(largest['community'])}, {largest['n_comments']:,} comments, scope '{largest['dominant_scope_by_comments']}', expected topic mix: {topic_mix_for(platform, int(largest['community']), n=2)}.")
    if not most_negative.empty:
        ci = community_uncertainty[
            (community_uncertainty["platform"] == platform)
            & (community_uncertainty["community"] == most_negative["community"])
        ]
        ci_text = ""
        if not ci.empty:
            ci = ci.iloc[0]
            ci_text = f"; author-level negative center {ci['author_pct_negative_center']:.1f}% with 95% CI {ci['pct_negative_lo']:.1f}-{ci['pct_negative_hi']:.1f}%."
        print(f"- Most negative report-grade community: c{int(most_negative['community'])}, {most_negative['pct_negative']:.1f}% comment-level negative{ci_text} Dominant topic '{most_negative['dominant_topic']}'.")
    if not most_mixed.empty:
        print(f"- Most topically mixed report-grade community: c{int(most_mixed['community'])}, entropy {most_mixed['topic_entropy']:.2f}, expected topic mix: {topic_mix_for(platform, int(most_mixed['community']), n=3)}.")

## Generated artifacts

Relevant existing outputs from this notebook are:

- `data/processed/07_community_sentiment_topics/comment_community_sentiment_topics.csv`
- `data/processed/07_community_sentiment_topics/community_coverage_summary.csv`
- `data/processed/07_community_sentiment_topics/community_threshold_summary.csv`
- `data/processed/07_community_sentiment_topics/community_threshold_justification.csv`
- `data/processed/07_community_sentiment_topics/community_sentiment_topic_summary.csv`
- `data/processed/07_community_sentiment_topics/community_report_grade_subset.csv`
- `data/processed/07_community_sentiment_topics/comment_community_report_grade_subset.csv`
- `data/processed/07_community_sentiment_topics/community_report_candidates.csv`
- `data/processed/07_community_sentiment_topics/community_atlas.csv`
- `data/processed/07_community_sentiment_topics/community_claim_safety.csv`
- `plots/community_coverage_funnel.png`
- `plots/community_bubble_atlas.png`

## Report-ready findings

- Raw Louvain community counts are not interpreted directly. The report interprets only communities passing the **100-comment** report-grade threshold.
- Coverage is much stronger on Reddit (**81.1%** of comments joined to author communities) than YouTube (**41.0%**), so YouTube community findings are conditional on reply-network participation.
- After the report-grade gate, Reddit retains **40** communities covering **88.4%** of assigned Reddit comments; YouTube retains **28** communities covering **52.8%** of assigned YouTube reply-network comments.
- The atlas and candidate tables provide interpretable examples, while claim-safety outputs guard against overstating small or low-coverage communities.
